<a href="https://colab.research.google.com/github/18217265596/sx/blob/master/LigandMPNN_Colab_Complete_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LigandMPNN → extract.py → ColabFold

流程：上传多链 PDB → LigandMPNN 设计 → `extract.py` 提取高分唯一序列 → 指定可选 de novo 链 → 非 de novo 链按唯一序列查询并复用 MMseqs2 MSA → AlphaFold2-Multimer v3（模型 1–3，无 Amber relaxation）→ 输出按 ipTM、缺失时按 pTM 排序的 CSV。ColabFold 结构与中间文件不会保留。


In [ ]:
# 0. 上传 PDB，并显示蛋白链顺序
from google.colab import files
from pathlib import Path
from collections import OrderedDict
import csv, hashlib, importlib.util, json, math, os, re, shutil, subprocess, sys

uploaded = files.upload()
items = [(n, b) for n, b in uploaded.items() if n.lower().endswith(".pdb")]
if len(items) != 1:
    raise ValueError("请一次只上传一个 .pdb 文件。")

ROOT = Path("/content/LigandMPNN")
INPUT_DIR = Path("/content/user_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_PDB = INPUT_DIR / Path(items[0][0]).name
USER_PDB.write_bytes(items[0][1])

AA3 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
    "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
    "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M",
}
residues = OrderedDict()
for line in USER_PDB.read_text(errors="replace").splitlines():
    if line[:6].strip() not in {"ATOM","HETATM"} or line[12:16].strip() != "CA":
        continue
    if line[16:17] not in {" ","A"}:
        continue
    resname = line[17:20].strip().upper()
    if resname not in AA3:
        continue
    chain = line[21:22].strip() or "_"
    key = (line[22:26].strip(), line[26:27].strip())
    residues.setdefault(chain, OrderedDict()).setdefault(key, AA3[resname])

PDB_CHAIN_SEQUENCES = OrderedDict(
    (chain, "".join(seq.values())) for chain, seq in residues.items() if seq
)
PDB_CHAIN_ORDER = list(PDB_CHAIN_SEQUENCES)
if not PDB_CHAIN_ORDER:
    raise ValueError("PDB 中没有识别到标准蛋白链 CA 原子。")

print("PDB:", USER_PDB)
for i, chain in enumerate(PDB_CHAIN_ORDER, 1):
    print(f"{i}. chain {chain}: {len(PDB_CHAIN_SEQUENCES[chain])} aa")


In [ ]:
# 1. 所有用户参数（de novo 链在下一个独立 cell）
CHECKPOINT_OPTIONS = {
 1:("protein_mpnn","proteinmpnn_v_48_002.pt","ProteinMPNN 0.02 Å"),
 2:("protein_mpnn","proteinmpnn_v_48_010.pt","ProteinMPNN 0.10 Å"),
 3:("protein_mpnn","proteinmpnn_v_48_020.pt","ProteinMPNN 0.20 Å"),
 4:("protein_mpnn","proteinmpnn_v_48_030.pt","ProteinMPNN 0.30 Å"),
 5:("ligand_mpnn","ligandmpnn_v_32_005_25.pt","LigandMPNN 0.05 Å"),
 6:("ligand_mpnn","ligandmpnn_v_32_010_25.pt","LigandMPNN 0.10 Å"),
 7:("ligand_mpnn","ligandmpnn_v_32_020_25.pt","LigandMPNN 0.20 Å"),
 8:("ligand_mpnn","ligandmpnn_v_32_030_25.pt","LigandMPNN 0.30 Å"),
 9:("per_residue_label_membrane_mpnn","per_residue_label_membrane_mpnn_v_48_020.pt","MembraneMPNN per-residue"),
 10:("global_label_membrane_mpnn","global_label_membrane_mpnn_v_48_020.pt","MembraneMPNN global"),
 11:("soluble_mpnn","solublempnn_v_48_002.pt","SolubleMPNN 0.02 Å"),
 12:("soluble_mpnn","solublempnn_v_48_010.pt","SolubleMPNN 0.10 Å"),
 13:("soluble_mpnn","solublempnn_v_48_020.pt","SolubleMPNN 0.20 Å"),
 14:("soluble_mpnn","solublempnn_v_48_030.pt","SolubleMPNN 0.30 Å"),
 15:("sidechain_packer","ligandmpnn_sc_v_32_002_16.pt","仅用于侧链打包"),
}
for i, (_, fn, desc) in CHECKPOINT_OPTIONS.items():
    print(f"{i:>2}: {fn:<48} | {desc}")

# LigandMPNN
TASK_CHECKPOINT_ID = 6
CHAINS_TO_DESIGN = "A"
SEED = 112
TEMPERATURE = 0.10
BATCH_SIZE = 10
NUMBER_OF_BATCHES = 10
PARSE_ATOMS_WITH_ZERO_OCCUPANCY = 1
SAVE_STATS = 1
FIXED_RESIDUES = ""
REDESIGNED_RESIDUES = ""
VERBOSE = 1
PACK_SIDE_CHAINS = False
NUMBER_OF_PACKS_PER_DESIGN = 1

# extract.py
EXTRACT_SOURCE_GLOB = "*.fa"
EXTRACT_TOP_N = 20
EXTRACT_COMBINED_FASTA_NAME = "all_sequences.fa"
EXTRACT_TSV_NAME = "top_unique_sequences.tsv"

# ColabFold
RUN_COLABFOLD = True
COLABFOLD_MSA_SERVER = "https://api.colabfold.com"
COLABFOLD_USE_ENV = True
COLABFOLD_USE_FILTER = True
COLABFOLD_MODEL_TYPE = "alphafold2_multimer_v3"
COLABFOLD_NUM_RECYCLES = 3
COLABFOLD_NUM_MODELS = 3
COLABFOLD_MODEL_ORDER = [1,2,3]
COLABFOLD_NUM_SEEDS = 1
COLABFOLD_USE_DROPOUT = False
COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE = "auto"
COLABFOLD_MAX_MSA = "auto"
COLABFOLD_NUM_RELAX = 0
COLABFOLD_CALC_EXTRA_PTM = True
COLABFOLD_MAX_BINDERS = None
COLABFOLD_JOB_PREFIX = "binder_complex"
COLABFOLD_FINAL_CSV_NAME = "colabfold_ranked_iptm_ptm.csv"
COLABFOLD_DELETE_INTERMEDIATES = True
DOWNLOAD_FINAL_CSV = True

if TASK_CHECKPOINT_ID not in range(1,15): raise ValueError("TASK_CHECKPOINT_ID 必须为 1–14。")
if FIXED_RESIDUES.strip() and REDESIGNED_RESIDUES.strip(): raise ValueError("FIXED_RESIDUES 与 REDESIGNED_RESIDUES 不能同时使用。")
if EXTRACT_TOP_N < 1: raise ValueError("EXTRACT_TOP_N 必须至少为 1。")
if COLABFOLD_MAX_BINDERS is not None and int(COLABFOLD_MAX_BINDERS) < 1: raise ValueError("COLABFOLD_MAX_BINDERS 必须为 None 或正整数。")
if COLABFOLD_MODEL_TYPE != "alphafold2_multimer_v3": raise ValueError("模型固定为 alphafold2_multimer_v3。")
if COLABFOLD_NUM_MODELS != 3 or COLABFOLD_MODEL_ORDER != [1,2,3]: raise ValueError("固定使用模型 1、2、3。")
if COLABFOLD_NUM_RELAX != 0: raise ValueError("当前流程不做 Amber relaxation。")
if not COLABFOLD_CALC_EXTRA_PTM: raise ValueError("请保持 COLABFOLD_CALC_EXTRA_PTM=True。")
if isinstance(COLABFOLD_NUM_RECYCLES,str) and COLABFOLD_NUM_RECYCLES != "auto": raise ValueError("NUM_RECYCLES 只能为整数或 auto。")
print("ColabFold recycles:", COLABFOLD_NUM_RECYCLES)


In [ ]:
# 2. 指定 de novo 链；留空表示所有链都做 MSA
COLABFOLD_DE_NOVO_CHAIN = "A"

COLABFOLD_DE_NOVO_CHAIN = COLABFOLD_DE_NOVO_CHAIN.strip()
if "," in COLABFOLD_DE_NOVO_CHAIN:
    raise ValueError("当前只支持一条 de novo 链。")
if COLABFOLD_DE_NOVO_CHAIN and COLABFOLD_DE_NOVO_CHAIN not in PDB_CHAIN_ORDER:
    raise ValueError(f"可用链：{PDB_CHAIN_ORDER}")
print("de novo chain:", COLABFOLD_DE_NOVO_CHAIN or "None（所有链均使用 MSA）")


In [ ]:
# 3. 安装 LigandMPNN、下载全部权重、修复兼容性
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(["git","clone","--depth","1","https://github.com/dauparas/LigandMPNN.git",str(ROOT)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade","ProDy==2.6.1","biopython>=1.81","ml-collections==0.1.1","dm-tree==0.1.8"],check=True)

MODEL_DIR = ROOT/"model_params"
MODEL_DIR.mkdir(parents=True,exist_ok=True)
subprocess.run(["bash",str(ROOT/"get_model_params.sh"),str(MODEL_DIR)],check=True)
downloaded = {p.name for p in MODEL_DIR.glob("*.pt")}
expected = {x[1] for x in CHECKPOINT_OPTIONS.values()}
missing = sorted(expected-downloaded)
if missing: raise RuntimeError("缺少权重："+", ".join(missing))
for p in MODEL_DIR.glob("*.pt"):
    if p.stat().st_size < 1024**2: raise RuntimeError(f"权重疑似不完整：{p}")

os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"]="1"
run_py=ROOT/"run.py"
s=run_py.read_text()
s=s.replace("torch.load(checkpoint_path, map_location=device)","torch.load(checkpoint_path, map_location=device, weights_only=False)")
s=s.replace("torch.load(args.checkpoint_path_sc, map_location=device)","torch.load(args.checkpoint_path_sc, map_location=device, weights_only=False)")
run_py.write_text(s)
if "checkpoint_path, map_location=device, weights_only=False" not in run_py.read_text(): raise RuntimeError("torch.load 补丁失败。")

aliases={r"\bnp\.int\b":"int",r"\bnp\.float\b":"float",r"\bnp\.bool\b":"bool",r"\bnp\.object\b":"object",r"\bnp\.str\b":"str",r"\bnp\.complex\b":"complex"}
for p in ROOT.rglob("*.py"):
    old=p.read_text(); new=old
    for a,b in aliases.items(): new=re.sub(a,b,new)
    if new!=old: p.write_text(new)

EXTRACT_PY=ROOT/"extract.py"
subprocess.run(["wget","-q","https://raw.githubusercontent.com/18217265596/sx/master/extract.py","-O",str(EXTRACT_PY)],check=True)
if not EXTRACT_PY.exists() or not EXTRACT_PY.stat().st_size: raise RuntimeError("extract.py 下载失败。")
print("LigandMPNN ready.")


In [ ]:
# 4. 运行 LigandMPNN，并用 extract.py 提取高分唯一复合物序列
MODEL_TYPE,CHECKPOINT_NAME,_=CHECKPOINT_OPTIONS[TASK_CHECKPOINT_ID]
CHECKPOINT_PATH=MODEL_DIR/CHECKPOINT_NAME
USER_OUT=ROOT/"outputs"/USER_PDB.stem
shutil.rmtree(USER_OUT,ignore_errors=True)
flags={"protein_mpnn":"--checkpoint_protein_mpnn","ligand_mpnn":"--checkpoint_ligand_mpnn","soluble_mpnn":"--checkpoint_soluble_mpnn","per_residue_label_membrane_mpnn":"--checkpoint_per_residue_label_membrane_mpnn","global_label_membrane_mpnn":"--checkpoint_global_label_membrane_mpnn"}
cmd=[sys.executable,"-u","run.py","--model_type",MODEL_TYPE,flags[MODEL_TYPE],str(CHECKPOINT_PATH),"--seed",str(SEED),"--pdb_path",str(USER_PDB),"--out_folder",str(USER_OUT),"--batch_size",str(BATCH_SIZE),"--number_of_batches",str(NUMBER_OF_BATCHES),"--temperature",str(TEMPERATURE),"--parse_atoms_with_zero_occupancy",str(PARSE_ATOMS_WITH_ZERO_OCCUPANCY),"--save_stats",str(SAVE_STATS),"--verbose",str(VERBOSE)]
if CHAINS_TO_DESIGN.strip(): cmd+=["--chains_to_design",CHAINS_TO_DESIGN.strip()]
if FIXED_RESIDUES.strip(): cmd+=["--fixed_residues",FIXED_RESIDUES.strip()]
if REDESIGNED_RESIDUES.strip(): cmd+=["--redesigned_residues",REDESIGNED_RESIDUES.strip()]
if PACK_SIDE_CHAINS: cmd+=["--pack_side_chains","1","--checkpoint_path_sc",str(MODEL_DIR/CHECKPOINT_OPTIONS[15][1]),"--number_of_packs_per_design",str(NUMBER_OF_PACKS_PER_DESIGN)]
env=os.environ.copy(); env["PYTHONUNBUFFERED"]="1"
r=subprocess.run(cmd,cwd=ROOT,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError("LigandMPNN 运行失败。")

SEQ_DIR=USER_OUT/"seqs"
fastas=sorted(SEQ_DIR.glob(EXTRACT_SOURCE_GLOB))
if not fastas: raise FileNotFoundError("未找到 LigandMPNN FASTA。")
combined=USER_OUT/EXTRACT_COMBINED_FASTA_NAME
with combined.open("w") as out:
    for f in fastas:
        t=f.read_text(); out.write(t); out.write("" if not t or t.endswith("\n") else "\n")
er=subprocess.run([sys.executable,str(EXTRACT_PY),"--input",str(combined),"--top",str(EXTRACT_TOP_N)],text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE)
if er.returncode: print(er.stderr); raise RuntimeError("extract.py 运行失败。")
(USER_OUT/EXTRACT_TSV_NAME).write_text(er.stdout)

records=[]
for line in er.stdout.splitlines():
    if line.strip():
        rank,conf,rid,total=line.split("\t",3)
        records.append({"rank":int(rank),"total_sequence":total.strip().upper()})
if COLABFOLD_MAX_BINDERS is not None: records=records[:int(COLABFOLD_MAX_BINDERS)]
if not records: raise RuntimeError("没有候选序列。")

designed={x.strip() for x in CHAINS_TO_DESIGN.split(",") if x.strip()} or set(PDB_CHAIN_ORDER)
if designed-set(PDB_CHAIN_ORDER): raise ValueError("CHAINS_TO_DESIGN 包含不存在的链。")
valid=set("ACDEFGHIKLMNPQRSTVWY")
CANDIDATES=[]
for rec in records:
    parts=[x.strip().upper() for x in rec["total_sequence"].split(":")]
    if len(parts)!=len(PDB_CHAIN_ORDER): raise ValueError(f"rank={rec['rank']} 链数与 PDB 不一致。")
    chainseq=OrderedDict(zip(PDB_CHAIN_ORDER,parts))
    for ch,seq in chainseq.items():
        if len(seq)!=len(PDB_CHAIN_SEQUENCES[ch]): raise ValueError(f"rank={rec['rank']} chain {ch} 长度不一致。")
        if set(seq)-valid: raise ValueError(f"rank={rec['rank']} chain {ch} 含非法字符。")
        if ch not in designed and seq!=PDB_CHAIN_SEQUENCES[ch]: raise ValueError(f"rank={rec['rank']} 固定 chain {ch} 与 PDB 不一致；请检查链顺序。")
    total=":".join(parts)
    job=f"{COLABFOLD_JOB_PREFIX}_{rec['rank']:03d}_{hashlib.sha1(total.encode()).hexdigest()[:8]}"
    CANDIDATES.append({"jobname":job,"total_sequence":total,"chain_sequences":chainseq,"de_novo_sequence":chainseq.get(COLABFOLD_DE_NOVO_CHAIN) if COLABFOLD_DE_NOVO_CHAIN else None})

unique_non_de_novo=OrderedDict()
for ch in PDB_CHAIN_ORDER:
    if ch==COLABFOLD_DE_NOVO_CHAIN: continue
    vals=OrderedDict((x["chain_sequences"][ch],None) for x in CANDIDATES)
    print(f"chain {ch}: {len(vals)} unique sequence(s) requiring MSA")
    for seq in vals: unique_non_de_novo.setdefault(seq,None)
print("Total unique MSA queries:",len(unique_non_de_novo))


In [ ]:
# 5. ColabFold：复用 MSA、运行模型 1–3、生成最终 CSV
if RUN_COLABFOLD:
    subprocess.run([sys.executable,"-m","pip","install","-q","--no-warn-conflicts","colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold"],check=True)
    subprocess.run(["bash","-lc","rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so /usr/local/lib/python3.*/dist-packages/tensorflow/lite/python/*/*.so 2>/dev/null || true"],check=True)

    spec=importlib.util.find_spec("colabfold.batch")
    if spec and spec.origin:
        p=Path(spec.origin); s=p.read_text()
        old='scores[k] = np.around(conf[-1][k], 2).item()'
        if old in s: p.write_text(s.replace(old,'scores[k] = float(conf[-1][k])'))

    from colabfold.colabfold import run_mmseqs2
    from colabfold.input import msa_to_str
    from colabfold.batch import run
    from colabfold.download import download_alphafold_params
    from colabfold.utils import setup_logging

    WORK=USER_OUT/"colabfold_work"; MSA_DIR=WORK/"msa"; PRED=WORK/"predictions"
    MSA_DIR.mkdir(parents=True,exist_ok=True); PRED.mkdir(parents=True,exist_ok=True)
    unique=list(unique_non_de_novo)
    MSA_BY_SEQUENCE={}
    if unique:
        digest=hashlib.sha1("\n".join(unique).encode()).hexdigest()[:12]
        msas=run_mmseqs2(unique,str(MSA_DIR/f"unique_{digest}"),use_env=COLABFOLD_USE_ENV,use_filter=COLABFOLD_USE_FILTER,use_templates=False,use_pairing=False,host_url=COLABFOLD_MSA_SERVER,user_agent="LigandMPNN-ColabFold-pipeline/1.0")
        if len(unique)==1 and isinstance(msas,str): msas=[msas]
        if len(msas)!=len(unique): raise RuntimeError("MSA 返回数量不一致。")
        MSA_BY_SEQUENCE=dict(zip(unique,msas))

    queries=[]
    for x in CANDIDATES:
        seqs=list(x["chain_sequences"].values())
        unpaired=[f">101\n{seq}\n" if ch==COLABFOLD_DE_NOVO_CHAIN else MSA_BY_SEQUENCE[seq] for ch,seq in x["chain_sequences"].items()]
        query_only=[f">101\n{seq}\n" for seq in seqs]
        a3m=msa_to_str(unpaired_msa=unpaired,paired_msa=query_only,query_seqs_unique=seqs,query_seqs_cardinality=[1]*len(seqs))
        queries.append((x["jobname"],seqs,[a3m],None))

    recycles=None if COLABFOLD_NUM_RECYCLES=="auto" else int(COLABFOLD_NUM_RECYCLES)
    tol=None if COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE=="auto" else float(COLABFOLD_RECYCLE_EARLY_STOP_TOLERANCE)
    max_msa=None if COLABFOLD_MAX_MSA=="auto" else COLABFOLD_MAX_MSA
    data=Path("/content/colabfold_params"); data.mkdir(exist_ok=True)
    setup_logging(WORK/"colabfold.log")
    download_alphafold_params(COLABFOLD_MODEL_TYPE,data)
    run(queries=queries,result_dir=PRED,use_templates=False,custom_template_path=None,num_relax=0,msa_mode="custom",model_type=COLABFOLD_MODEL_TYPE,num_models=3,num_recycles=recycles,relax_max_iterations=0,recycle_early_stop_tolerance=tol,num_seeds=COLABFOLD_NUM_SEEDS,use_dropout=COLABFOLD_USE_DROPOUT,model_order=[1,2,3],is_complex=True,data_dir=data,keep_existing_results=False,rank_by="auto",pair_mode="unpaired",pairing_strategy="greedy",stop_at_score=100.0,prediction_callback=None,input_features_callback=None,dpi=100,zip_results=False,save_all=False,max_msa=max_msa,use_cluster_profile=not("multimer" in COLABFOLD_MODEL_TYPE and max_msa is not None),save_recycles=False,user_agent="LigandMPNN-ColabFold-pipeline/1.0",calc_extra_ptm=True,skip_output=["plots","pae_json","msa"])

    score_files=sorted(PRED.rglob("*scores*.json"))
    rows=[]
    for x in CANDIDATES:
        scores=[]
        for f in score_files:
            if f.name.startswith(x["jobname"]) or x["jobname"] in str(f.parent):
                d=json.loads(f.read_text())
                def val(k):
                    try:
                        z=float(d.get(k)); return z if math.isfinite(z) else None
                    except (TypeError,ValueError): return None
                iptm,ptm=val("iptm"),val("ptm")
                scores.append((iptm if iptm is not None else ptm,iptm,ptm))
        if scores:
            _,iptm,ptm=max(scores,key=lambda z: z[0] if z[0] is not None else float("-inf"))
        else:
            iptm=ptm=None
            print("Warning: no scores for",x["jobname"])
        rows.append({"de_novo_sequence":x["de_novo_sequence"],"total_sequence":x["total_sequence"],"iptm":iptm,"ptm":ptm,"sort":iptm if iptm is not None else ptm})
    rows.sort(key=lambda x:(x["sort"] is not None,x["sort"] if x["sort"] is not None else float("-inf")),reverse=True)

    FINAL_CSV=USER_OUT/COLABFOLD_FINAL_CSV_NAME
    fields=["de_novo_sequence","total_sequence","iptm","ptm"] if COLABFOLD_DE_NOVO_CHAIN else ["total_sequence","iptm","ptm"]
    with FINAL_CSV.open("w",newline="",encoding="utf-8") as h:
        w=csv.DictWriter(h,fieldnames=fields); w.writeheader()
        for row in rows:
            out={k:row.get(k) for k in fields}
            for k in ("iptm","ptm"): out[k]="" if out[k] is None else f"{out[k]:.6f}"
            w.writerow(out)
    print(FINAL_CSV.read_text()[:5000])
    if COLABFOLD_DELETE_INTERMEDIATES: shutil.rmtree(WORK,ignore_errors=True)
    if DOWNLOAD_FINAL_CSV: files.download(str(FINAL_CSV))


## CSV 规则

- 指定 `COLABFOLD_DE_NOVO_CHAIN`：输出 `de_novo_sequence,total_sequence,iptm,ptm`。
- 留空：输出 `total_sequence,iptm,ptm`。
- `total_sequence` 按输入 PDB 的蛋白链顺序用 `:` 连接。
- 每条候选从模型 1–3 中选择 ipTM 最高者；若均无 ipTM，则按 pTM 选择。
- 整个 CSV 按 ipTM 降序；缺失 ipTM 的行用 pTM 作为排序值。
